In [1]:
pip install nilearn scikit-learn pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from nilearn import datasets

print("⏳ Conectando con ABIDE y descargando datos base...")

# Descarga limpia y oficial para versiones modernas de Nilearn
data_triplet = datasets.fetch_abide_pcp(
    n_subjects=40,
    quality_checked=True
)

print("✅ ¡Descarga e indexación completadas con éxito!")

⏳ Conectando con ABIDE y descargando datos base...


[fetch_abide_pcp] Dataset found in C:\Users\luna_\nilearn_data\ABIDE_pcp

✅ ¡Descarga e indexación completadas con éxito!


In [3]:
# utilizamos pandas para crear el data set con datos clinicos
phenotypic_data = pd.DataFrame(data_triplet.phenotypic)

# 2. Transformamos DX_GROUP: 1 queda como 1 (Autismo), 2 pasa a ser 0 (Control)
y = np.where(phenotypic_data['DX_GROUP'] == 1, 1, 0)

# 3. Revisamos cómo quedó distribuido nuestro piloto
print("📊 DISTRIBUCIÓN DEL GRUPO PILOTO:")
print(f"Total de pacientes analizados: {len(y)}")
print(f"🧩 Pacientes dentro del espectro (Clase 1): {sum(y)}")
print(f"🧠 Pacientes de control neurotípicos (Clase 0): {len(y) - sum(y)}")

# Miramos las primeras filas de la tabla clínica para chusmear qué otros datos hay (edad, sexo, etc.)
phenotypic_data[['SUB_ID', 'X', 'AGE_AT_SCAN', 'SEX', 'DX_GROUP']].head(100)

📊 DISTRIBUCIÓN DEL GRUPO PILOTO:
Total de pacientes analizados: 40
🧩 Pacientes dentro del espectro (Clase 1): 21
🧠 Pacientes de control neurotípicos (Clase 0): 19


,SUB_ID,X,AGE_AT_SCAN,SEX,DX_GROUP
1,50003,2,24.45,1,1
2,50004,3,19.09,1,1
3,50005,4,13.73,2,1
4,50006,5,13.37,1,1
5,50007,6,17.78,1,1
6,50008,7,32.45,1,1
8,50010,9,35.20,1,1
9,50011,10,16.93,1,1
10,50012,11,21.48,1,1
11,50013,12,9.33,1,1


In [4]:
# Miramos qué variables reales se crearon dentro de tu descarga limpia
print("📋 CLAVES REALES DISPONIBLES EN TU OBJETO:")
keys_reales = list(data_triplet.keys())
print(keys_reales)

# Buscamos de forma segura si hay algo que contenga la palabra 'rois' o 'func'
encontrados = [k for k in keys_reales if 'roi' in k or 'func' in k]
print(f"\n📌 Variables de mapas cerebrales detectadas: {encontrados}")

📋 CLAVES REALES DISPONIBLES EN TU OBJETO:
['description', 'phenotypic', 'func_preproc']

📌 Variables de mapas cerebrales detectadas: ['func_preproc']


In [5]:
import os
import warnings
from nilearn.maskers import NiftiLabelsMasker
from nilearn import datasets as nilearn_datasets
from nilearn.connectome import ConnectivityMeasure
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Silenciamos advertencias futuras para que la consola quede limpia
warnings.filterwarnings("ignore", category=FutureWarning)

print("🗺️ 1. Cargando el atlas cerebral Harvard-Oxford...")
atlas = nilearn_datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
atlas_filename = atlas.maps

# Usamos 'zscore_sample' directo para evitar que Nilearn se queje
masker = NiftiLabelsMasker(labels_img=atlas_filename, standardize='zscore_sample', memory="nilearn_cache")

print("\n⏳ 2. Extrayendo señales numéricas (Procesando escaneos 3D)...")
time_series_list = []
total_pacientes = len(data_triplet.func_preproc)

for i, archivo_cerebro in enumerate(data_triplet.func_preproc):
    # Imprime el progreso en la misma línea
    print(f" 🧩 Procesando paciente {i+1}/{total_pacientes}...", end="\r")
    signals = masker.fit_transform(archivo_cerebro)
    time_series_list.append(signals)

print("\n\n🧠 3. Calculando matrices de conectividad funcional (X)...")
conn_measure = ConnectivityMeasure(kind='correlation', vectorize=True)
X = conn_measure.fit_transform(time_series_list)
print(f"📐 Tamaño final de la matriz X: {X.shape}")

# EXTRA: Guardamos X e y localmente en tu carpeta para no tener que recalcular nunca más
np.save('X_conectividad.npy', X)
np.save('y_diagnostico.npy', y)
print("💾 ¡Matrices respaldadas localmente como archivos .npy!")

print("\n✂️ 4. Dividiendo el dataset (80% Train / 20% Test)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("⚖️ 5. Escalando características...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("🤖 6. Entrenando el modelo de Machine Learning (Regresión Logística)...")
model = LogisticRegression(C=0.1, penalty='l2', max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("\n🎯 --- RESULTADOS DEL MODELO PILOTO ---")
print(f"Precisión General (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\n📝 Reporte de Clasificación Detallado:")
print(classification_report(y_test, y_pred, target_names=['Control (0)', 'Autismo (1)']))

🗺️ 1. Cargando el atlas cerebral Harvard-Oxford...


[fetch_atlas_harvard_oxford] Dataset found in C:\Users\luna_\nilearn_data\fsl


⏳ 2. Extrayendo señales numéricas (Procesando escaneos 3D)...
 🧩 Procesando paciente 40/40...

🧠 3. Calculando matrices de conectividad funcional (X)...
📐 Tamaño final de la matriz X: (40, 1176)
💾 ¡Matrices respaldadas localmente como archivos .npy!

✂️ 4. Dividiendo el dataset (80% Train / 20% Test)...
⚖️ 5. Escalando características...
🤖 6. Entrenando el modelo de Machine Learning (Regresión Logística)...

🎯 --- RESULTADOS DEL MODELO PILOTO ---
Precisión General (Accuracy): 62.50%

📝 Reporte de Clasificación Detallado:
              precision    recall  f1-score   support

 Control (0)       0.57      1.00      0.73         4
 Autismo (1)       1.00      0.25      0.40         4

    accuracy                           0.62         8
   macro avg       0.79      0.62      0.56         8
weighted avg       0.79      0.62      0.56         8

